# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krashishkr008-ghg/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why
## Method Choice

I chose a Random Forest Regressor because my lane is Search Performance Optimization, where the goal is to predict search performance (CTR) from safe features.

Random Forest can learn non-linear relationships between search volume, average position, and other search metrics while reducing overfitting through multiple decision trees.

This model will be compared with my Week-4 baseline on the same dataset and split. The results will be used only for decision-support.


In [10]:
import pandas as pd

# Load dataset
df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nFeatures selected for modeling:")
features = ["search_volume", "avg_position"]

for f in features:
    print("-", f)

print("\nTarget:")
print("- ctr")

Rows: 30000
Columns: 44

Features selected for modeling:
- search_volume
- avg_position

Target:
- ctr


## 2. ## Split Design

I use a grouped train/test split based on `client_id`.

Grouping by client prevents the model from seeing data from the same client in both the training and testing sets. This makes the evaluation more realistic because the model is tested on unseen clients instead of memorizing patterns from clients it has already seen.

The split is honest for my Search Performance Optimization lane because the same grouped split is used for both the baseline and the machine learning model. The evaluation is intended for decision-support and avoids data leakage between clients.

In [11]:
from sklearn.model_selection import GroupShuffleSplit

# Features
X = df[["search_volume", "avg_position"]].fillna(0)

# Target
y = df["ctr"]

# Group by client
groups = df["client_id"]

# Grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))
print("Training clients:", df.iloc[train_idx]["client_id"].nunique())
print("Testing clients :", df.iloc[test_idx]["client_id"].nunique())
print("Shared clients  :", len(set(df.iloc[train_idx]["client_id"]) &
                                set(df.iloc[test_idx]["client_id"])))

Training samples: 23837
Testing samples : 6163
Training clients: 25
Testing clients : 7
Shared clients  : 0


## 3. ## Train + Compare vs My Baseline

I trained a Random Forest Regressor to predict CTR using the same grouped train/test split that I defined above.

The Week-4 baseline is a simple predictor that uses the mean CTR from the training data. Both the baseline and the Random Forest are evaluated on the same testing set.

The comparison uses the same evaluation metrics (Mean Absolute Error and R² score) to provide a fair comparison.

The results are intended for decision-support rather than automated decision-making.

In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

# -------------------------
# Train Random Forest model
# -------------------------
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Model predictions
rf_predictions = model.predict(X_test)

# -------------------------
# Week-4 Baseline
# -------------------------
baseline_predictions = [y_train.mean()] * len(y_test)

# -------------------------
# Metrics
# -------------------------
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
rf_mae = mean_absolute_error(y_test, rf_predictions)

baseline_r2 = r2_score(y_test, baseline_predictions)
rf_r2 = r2_score(y_test, rf_predictions)

# -------------------------
# Comparison Table
# -------------------------
comparison = pd.DataFrame({
    "Model": ["Week-4 Baseline", "Random Forest"],
    "MAE": [round(baseline_mae, 4), round(rf_mae, 4)],
    "R² Score": [round(baseline_r2, 4), round(rf_r2, 4)]
})

print("Model vs Baseline Comparison")
display(comparison)

Model vs Baseline Comparison


,Model,MAE,R² Score
0,Week-4 Baseline,0.5537,-0.0657
1,Random Forest,0.6032,-1.3613


## 4. ## Errors and Interpretation

The Random Forest model performed worse than the Week-4 baseline on the grouped test split.

The model appears to make larger errors when predicting CTR for clients that were not seen during training. This suggests that the selected features do not capture all of the factors that influence CTR.

The model mainly relies on search volume and average position because these are the only features used during training.

This result suggests that additional safe features and feature engineering may improve performance. These findings are intended for decision-support rather than automated decisions.

In [13]:
# Feature importance and short error analysis

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print("Feature Importance")
display(importance)

print("\nShort Error Analysis")
print("- The model performs worse than the Week-4 baseline on unseen clients.")
print("- It mainly relies on search_volume and avg_position.")
print("- Some prediction errors may occur because important CTR-related signals are not included.")
print("- Additional safe features could improve the model.")

Feature Importance


,Feature,Importance
1,avg_position,0.833775
0,search_volume,0.166225



Short Error Analysis
- The model performs worse than the Week-4 baseline on unseen clients.
- It mainly relies on search_volume and avg_position.
- Some prediction errors may occur because important CTR-related signals are not included.
- Additional safe features could improve the model.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Committed to my repo under work/notebooks/ — then submit my repo URL on the card.